# SAIGE SINGLE-VARIANT TESTS 


In [ ]:
# Imports the 'os' module, which allows interaction with the operating system (e.g., environment variables, file paths).
import os 

# Retrieves the value of the environment variable 'WORKSPACE_BUCKET'.
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Displays the value of 'my_bucket' (i.e., the workspace bucket URL or path).
my_bucket


# COMMAND 

In [ ]:
%%writefile biosample_age_single_assoc.sh

#!/bin/bash
set -o errexit
set -o nounset

# Loop through the populations
for pop in EUR; do

    # Loop through the phenotypes
    for pheno in biosample_age; do
   
        echo "Processing pop: ${pop} | phenotype: ${pheno} | chr: ${chrom}" 
        
        step2_SPAtests.R \
            --bedFile="${plink_bed}" \
            --bimFile="${plink_bim}" \
            --famFile="${plink_fam}" \
            --SAIGEOutputFile="${output}/${pop}_${pheno}_single_assoc_chr${chrom}" \
            --LOCO=FALSE \
            --AlleleOrder=alt-first \
            --minMAF=0 \
            --minMAC=0.5 \
            --maxMAF=0.015 \
            --GMMATmodelFile="${null_glmm}/${pop}_${pheno}_null_glmm.rda" \
            --varianceRatioFile="${null_glmm}/${pop}_${pheno}_null_glmm.varianceRatio.txt" \
            --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
            --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
            --is_overwrite_output=TRUE \
            --is_Firth_beta=TRUE \
            --pCutoffforFirth=0.05 \
            --is_output_moreDetails=TRUE \
            --max_MAC_for_ER=10 


    done
done


In [ ]:
# Save script to workspace bucket 
!gsutil cp biosample_age_single_assoc.sh {my_bucket}/data/scripts/

In [ ]:
# Check script in workspace bucket 
!gsutil cat {my_bucket}/data/scripts/biosample_age_single_assoc.sh

# SUBMISSION 

In [ ]:
%%bash --out saige_gene_step02

# Loop through each chromosome 
for chrom 2 4 15 17; do

  # Source the dsub configuration file
  source ~/aou_dsub.bash 

  # Submit a dsub command 
  aou_dsub \
    --image ${ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2 \
    --boot-disk-size 10 \
    --disk-size 20 \
    --min-ram 4 \
    --logging "${WORKSPACE_BUCKET}/data/logging" \
    --input plink_bed="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bed" \
    --input plink_bim="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.bim" \
    --input plink_fam="${WORKSPACE_BUCKET}/data/saige_step01/lovo_files/EUR_LOVO_exome_chr${chromo}.fam" \
    --input-recursive grm="${WORKSPACE_BUCKET}/data/saige_step00/grm/" \
    --input-recursive null_glmm="${WORKSPACE_BUCKET}/data/saige_step01/null_glmm/" \
    --output-recursive output="${WORKSPACE_BUCKET}/data/saige_step02/age_tests/" \
    --env chrom=${chromo} \
    --script "${WORKSPACE_BUCKET}/data/scripts/biosample_age_single_assoc.sh"

done